In [ ]:
!pip install -q langchain
!pip install -q chromadb
!pip install -q pypdf
!pip install -q langchain-community
!pip install -q langchain-google-genai
!pip install -q langchain-text-splitters
!pip install -U langchain langchain-community langchain-text-splitters -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.9/132.9 kB 3.8 MB/s eta 0:00:00


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving IEFT MODULE 1.pdf to IEFT MODULE 1.pdf


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("IEFT MODULE 1.pdf")
docs = loader.load()

print("Pages loaded:", len(docs))

Pages loaded: 72


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(docs)

print("Total chunks created:", len(chunks))

Total chunks created: 77


In [ ]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_KEY")
print("API key set successfully")

API key set successfully


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

print("Embedding model loaded successfully")

Embedding model loaded successfully


In [ ]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)



print("Vector database created and saved successfully")

Vector database created and saved successfully


In [ ]:
query = "What is Economics?"

docs = vectorstore.similarity_search(query, k=3)

for doc in docs:
    print(doc.page_content[:500])

Economics
• Economics originated from the Greek word 
‘IOKONOMIA’ which means Household 
Management
• Adam Smith ( Father of Economics) > 1776 > 
‘Wealth of Nations’  
• Economics studies how the society and 
individuals use the limited resources to satisfy 
the unlimited wants. 
Hingston Xavier, Assistant Professor - Christ 
College of Engg
Economics
• Economics originated from the Greek word 
‘IOKONOMIA’ which means Household 
Management
• Adam Smith ( Father of Economics) > 1776 > 
‘Wealth of Nations’  
• Economics studies how the society and 
individuals use the limited resources to satisfy 
the unlimited wants. 
Hingston Xavier, Assistant Professor - Christ 
College of Engg
Scarcity and Choice 
• Scarcity means that resources are not available in 
the required quantity to satisfy all the wants and 
needs. 
• Since we face Scarcity, people have to make 
choice between goods and services. 
• In 1932, Lionnel Robinson ( ‘Nature and 
significance of Economics’) defined economics as 
a

In [ ]:
!pip install groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.6 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
import os
from groq import Groq

try:
    os.environ["GROQ_KEY"] = userdata.get("GROQ_KEY")

    client = Groq(
        api_key=os.environ["GROQ_KEY"]
    )

    print("Groq connected successfully")

except Exception as e:
    print("Error:", e)

Groq connected successfully


In [ ]:
def ask(question):
    try:
        # Retrieve relevant chunks
        docs = vectorstore.similarity_search(
            question,
            k=3
        )

        if not docs:
            return "No relevant information found."

        context = "\n\n".join(
            [doc.page_content for doc in docs]
        )

        prompt = f"""
Use the provided context to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        return response.choices[0].message.content

    except Exception as e:
        return f"Error: {str(e)}"

In [ ]:
print(ask("What is Economics?"))

Economics is the study of how society and individuals use limited resources to satisfy their unlimited wants. It is also defined as a "science which studies human behaviour in relationship with given ends and scarce means" by Lionnel Robinson in 1932.


In [ ]:
questions = [
    "What is Economics?",
    "Define demand",
    "What are the factors of production?",
    "What is scarcity?"
]

for q in questions:
    print("\n" + "="*50)
    print("Question:", q)
    print("Answer:", ask(q))


Question: What is Economics?
Answer: Economics is the study of how society and individuals use limited resources to satisfy their unlimited wants. It is also defined as a "science which studies human behaviour in relationship with given ends and scarce means" by Lionnel Robinson in 1932.

Question: Define demand
Answer: Demand is the desire backed by the ability and willingness to pay for a commodity.

Question: What are the factors of production?
Answer: The factors of production are:

1. Land (La) - Surface soil + Natural resources
2. Labour (L) - Mentally and Physically fit for work
3. Capital (K) - All Man made aids to production
4. Organization / Entrepreneurship - Combines all factors of production

Question: What is scarcity?
Answer: Scarcity means that resources are not available in the required quantity to satisfy all the wants and needs.


In [ ]:
def ask(question):
    docs = vectorstore.similarity_search(question, k=3)

    print("\nRetrieved Chunks:")
    for i, doc in enumerate(docs, 1):
        print(f"\nChunk {i}:")
        print(doc.page_content[:300])

    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""
Use the provided context to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [ ]:
print(ask("What is Economics?"))


Retrieved Chunks:

Chunk 1:
Economics
• Economics originated from the Greek word 
‘IOKONOMIA’ which means Household 
Management
• Adam Smith ( Father of Economics) > 1776 > 
‘Wealth of Nations’  
• Economics studies how the society and 
individuals use the limited resources to satisfy 
the unlimited wants. 
Hingston Xavier, As

Chunk 2:
Economics
• Economics originated from the Greek word 
‘IOKONOMIA’ which means Household 
Management
• Adam Smith ( Father of Economics) > 1776 > 
‘Wealth of Nations’  
• Economics studies how the society and 
individuals use the limited resources to satisfy 
the unlimited wants. 
Hingston Xavier, As

Chunk 3:
Scarcity and Choice 
• Scarcity means that resources are not available in 
the required quantity to satisfy all the wants and 
needs. 
• Since we face Scarcity, people have to make 
choice between goods and services. 
• In 1932, Lionnel Robinson ( ‘Nature and 
significance of Economics’) defined eco
Economics studies how the society and individual